# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Listing all found `RecordSet` entities and their constituent `Field` objects for further extraction. All entities are referenced by their `@id`.

In [ ]:
# Get an overview of the record sets and their fields
record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record sets.")
for rs in record_sets:
    print(f"Record set name: {rs.name}  (@id: {rs.id})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, Data type: {field.data_type})")
    print("  Columns:")
    for col in getattr(rs, 'columns', []):
        print(f"    - {col.name} (@id: {col.id}, Source: {col.source})")
    print('-' * 40)

# For illustration, select the main record set (assume only one main data table)
if len(record_sets):
    main_record_set_id = record_sets[0].id  # Use the first found record set
    print(f"Main record set chosen for analysis: {main_record_set_id}")
else:
    raise Exception('No record sets found in metadata -- check schema or dataset.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"  (No records found for {record_set_id})")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {list(df.columns)}")
    print(f"  Head:\n{df.head(2)}\n")

# Select the main record set for further EDA
main_df = dataframes[main_record_set_id]
print(f"DataFrame columns for {main_record_set_id}: {main_df.columns.tolist()}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, and grouping data by key attributes to prepare it for further analysis.

**Choose numeric and group fields for demonstration based on the dataset's schema overview above.**

In [ ]:
# Select a numeric field for analysis
# For demonstration, attempt to select a column containing 'coverage', 'MW', or 'peptide' in its name.
import numpy as np

numeric_candidates = [col for col in main_df.columns if any(word in col.lower() for word in ['coverage', 'mw', 'peptide', 'abundance'])]
if not numeric_candidates:
    numeric_field = main_df.select_dtypes(include=[np.number]).columns[0]
else:
    numeric_field = numeric_candidates[0]

print(f"Chosen numeric field for EDA: {numeric_field}")

threshold = main_df[numeric_field].mean() if np.issubdtype(main_df[numeric_field].dtype, np.number) else 10
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (found {filtered_df.shape[0]} records):")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a candidate group field (e.g., a 'sample' or 'description' column if present)
group_candidates = [col for col in main_df.columns if any(word in col.lower() for word in ['sample', 'description', 'accession', 'group'])]

group_field = group_candidates[0] if group_candidates else None
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the chosen numeric field
plt.figure(figsize=(8, 5))
sns.histplot(main_df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If there is a normalization column, plot vs group if present
if group_field and group_field in filtered_df.columns:
    top_groups = filtered_df[group_field].value_counts().index[:5]
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field, y=f"{numeric_field}_normalized", data=filtered_df[filtered_df[group_field].isin(top_groups)])
    plt.title(f"Normalized {numeric_field} by {group_field} (top 5)")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


- We demonstrated loading and exploring the [FAIR^2 Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) Croissant dataset package.
- Examined available record sets and their fields using their `@id` for robust schema-based data investigation.
- Loaded records, performed basic filtering, normalization, and group-based aggregation, and visualized field distributions.
- This notebook can be further extended for domain-specific analyses or integration with other `mlcroissant`-compatible biomedical datasets.
